# MIMIC-CXR Data Preprocessing Pipeline

**Mục tiêu:** Tiền xử lý dữ liệu MIMIC-CXR (ảnh X-quang + báo cáo y tế) cho subset `p10`, sau đó tạo split train/val/test và các CSV dùng cho training.

**Datasets sử dụng (Kaggle):**
- `phuong20052/mimic-cxr-jpg-lite` - Ảnh X-quang phổi (JPG)
- `phuong20052/mimic-cxr-reported` - Báo cáo y tế (text/CSV)

**Lưu ý:** `p10` là patient group/folder của MIMIC-CXR, tương ứng với `subject_id` bắt đầu bằng `10`. `study_id` vẫn là các mã dạng số trong thư mục `s<study_id>`.

**Pipeline gồm 8 bước chính:**
1. Setup môi trường và khám phá dữ liệu
2. Trích xuất section FINDINGS / IMPRESSION từ báo cáo thô hoặc CSV đã tiền xử lý
3. Làm sạch văn bản
4. Ghép cặp ảnh và báo cáo
5. Lọc dữ liệu
6. Kiểm tra ảnh và chuẩn hóa metadata
7. Xây vocabulary/tokenize
8. Chia train/val/test, kiểm tra phân bố nhãn và lưu kết quả


## 1. Setup môi trường & Import thư viện

In [ ]:
import os
import re
import json
import pickle
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

# Sklearn
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
tqdm.pandas()

# Set random seed để đảm bảo reproducibility
SEED = 42
np.random.seed(SEED)

# Cấu hình hiển thị
pd.set_option('display.max_colwidth', 200)
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print('Thư viện đã được import thành công')


In [ ]:
# Cấu hình đường dẫn dataset trên Kaggle
# Lưu ý: trên Kaggle, dataset có thể được mount trực tiếp dưới /kaggle/input
# hoặc dưới /kaggle/input/datasets/<username>/<dataset-slug>.

PATIENT_GROUP = 'p10'
SUBJECT_PREFIX = PATIENT_GROUP.lstrip('p')

def find_dataset_path(slug, extra_candidates=None):
    candidates = [Path(f'/kaggle/input/{slug}')]
    if extra_candidates:
        candidates.extend(Path(p) for p in extra_candidates)

    for path in candidates:
        if path.exists():
            return path

    input_root = Path('/kaggle/input')
    if input_root.exists():
        matches = sorted(input_root.glob(f'**/{slug}'))
        for path in matches:
            if path.is_dir():
                return path

    return candidates[0]

def resolve_files_root(dataset_path, patient_group=PATIENT_GROUP):
    candidates = [
        dataset_path,
        dataset_path / 'files',
        dataset_path / 'mimic-cxr-reports',
        dataset_path / 'mimic-cxr-reports' / 'files',
    ]
    for root in candidates:
        if (root / patient_group).exists():
            return root

    if dataset_path.exists():
        matches = sorted(dataset_path.glob(f'**/{patient_group}'))
        for match in matches:
            if match.is_dir():
                return match.parent

    return dataset_path

def filter_to_patient_group(df, patient_group=PATIENT_GROUP):
    if 'subject_id' not in df.columns:
        return df.copy()

    prefix = patient_group.lstrip('p')
    out = df[df['subject_id'].astype(str).str.startswith(prefix)].copy()
    return out

def add_image_paths(df, image_files_root, patient_group=PATIENT_GROUP):
    if {'dicom_id', 'subject_id', 'study_id'}.issubset(df.columns):
        out = df.copy()
        out['image_path'] = out.apply(
            lambda row: str(
                image_files_root
                / patient_group
                / f'p{int(row["subject_id"])}'
                / f's{int(row["study_id"])}'
                / f'{row["dicom_id"]}.jpg'
            ),
            axis=1,
        )
        return out
    return df.copy()

def find_csv_recursive(base_path, filenames):
    for name in filenames:
        direct = base_path / name
        if direct.exists():
            return direct

    if base_path.exists():
        for name in filenames:
            matches = sorted(base_path.glob(f'**/{name}'))
            if matches:
                return matches[0]

    return None

IMAGE_DATASET_PATH = find_dataset_path(
    'mimic-cxr-jpg-lite',
    extra_candidates=['/kaggle/input/datasets/phuong20052/mimic-cxr-jpg-lite'],
)
REPORT_DATASET_PATH = find_dataset_path(
    'mimic-cxr-reported',
    extra_candidates=[
        '/kaggle/input/mimic-cxr-reports',
        '/kaggle/input/datasets/phuong20052/mimic-cxr-reported',
    ],
)

IMAGE_FILES_ROOT = resolve_files_root(IMAGE_DATASET_PATH, PATIENT_GROUP)
REPORT_FILES_ROOT = resolve_files_root(REPORT_DATASET_PATH, PATIENT_GROUP)
PATIENT_GROUP_IMAGE_PATH = IMAGE_FILES_ROOT / PATIENT_GROUP
PATIENT_GROUP_REPORT_PATH = REPORT_FILES_ROOT / PATIENT_GROUP

# Thư mục output mới chỉ dành cho subset p10
OUTPUT_DIR = Path('/kaggle/working/processed') / PATIENT_GROUP
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Patient group: {PATIENT_GROUP} (subject_id bắt đầu bằng {SUBJECT_PREFIX})')
print(f'IMAGE_DATASET_PATH: {IMAGE_DATASET_PATH}')
print(f'IMAGE_FILES_ROOT: {IMAGE_FILES_ROOT}')
print(f'REPORT_DATASET_PATH: {REPORT_DATASET_PATH}')
print(f'REPORT_FILES_ROOT: {REPORT_FILES_ROOT}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

print('\nCấu trúc dataset ảnh:')
if IMAGE_DATASET_PATH.exists():
    for item in list(IMAGE_DATASET_PATH.iterdir())[:10]:
        print(f'  - {item.name}')
else:
    print('Đường dẫn ảnh chưa tồn tại - hãy add dataset vào notebook')

print('\nCấu trúc dataset report:')
if REPORT_DATASET_PATH.exists():
    for item in list(REPORT_DATASET_PATH.iterdir())[:10]:
        print(f'  - {item.name}')
else:
    print('Đường dẫn report chưa tồn tại - hãy add dataset vào notebook')


## 2. Khám phá dữ liệu (Exploratory Data Analysis)

MIMIC-CXR có cấu trúc thư mục dạng phân cấp:
```
files/
├── p10/ # patient group (subject_id bắt đầu bằng 10)
│ └── p10000032/ # subject_id
│ ├── s50414267/ # study_id
│ │ ├── <dicom_id>.jpg
│ │ └── <dicom_id>.jpg
│ └── s50414267.txt # báo cáo y tế
```

**Khóa định danh quan trọng:**
- `subject_id` — bệnh nhân
- `study_id` — một lượt chụp (có thể có nhiều ảnh)
- `dicom_id` — định danh ảnh

In [ ]:
# 2.1. Load metadata file
# File metadata gốc: mimic-cxr-2.0.0-metadata.csv
# Cột quan trọng: dicom_id, subject_id, study_id, ViewPosition

metadata_path = find_csv_recursive(
    IMAGE_DATASET_PATH,
    ['mimic-cxr-2.0.0-metadata.csv', 'metadata.csv', 'mimic-cxr-metadata.csv'],
)

if metadata_path:
    metadata_all = pd.read_csv(metadata_path)
    metadata = filter_to_patient_group(metadata_all, PATIENT_GROUP)
    metadata = add_image_paths(metadata, IMAGE_FILES_ROOT, PATIENT_GROUP)

    print(f'Đã load metadata từ: {metadata_path}')
    print(f'   Số dòng gốc: {len(metadata_all):,}')
    print(f'   Số dòng sau khi lọc {PATIENT_GROUP}: {len(metadata):,}')
    print(f'   Cột: {list(metadata.columns)[:12]}')
    display(metadata.head())
else:
    print('Không tìm thấy file metadata - sẽ tự xây dựng từ cấu trúc thư mục')
    metadata = None


In [ ]:
# 2.2. Nếu không có metadata, scan thư mục p10 để lấy danh sách ảnh

def scan_image_directory(base_path, limit=None):
    """Quét cấu trúc thư mục để xây dựng DataFrame ảnh.

    Mỗi dòng tương ứng với một file .jpg, kèm subject_id và study_id.
    """
    records = []
    image_files = list(base_path.rglob('*.jpg'))

    if limit:
        image_files = image_files[:limit]

    for img_path in tqdm(image_files, desc='Quét ảnh'):
        try:
            # Cấu trúc: .../p<group>/p<subject_id>/s<study_id>/<dicom_id>.jpg
            parts = img_path.parts
            study_dir = parts[-2]   # s<study_id>
            subject_dir = parts[-3] # p<subject_id>

            records.append({
                'dicom_id': img_path.stem,
                'subject_id': int(subject_dir.lstrip('p')),
                'study_id': int(study_dir.lstrip('s')),
                'image_path': str(img_path)
            })
        except (ValueError, IndexError):
            continue

    return pd.DataFrame(records)

if metadata is None:
    metadata = scan_image_directory(PATIENT_GROUP_IMAGE_PATH)
else:
    metadata = filter_to_patient_group(metadata, PATIENT_GROUP)
    if 'image_path' not in metadata.columns:
        metadata = add_image_paths(metadata, IMAGE_FILES_ROOT, PATIENT_GROUP)

print(f'\nTổng số ảnh trong {PATIENT_GROUP}: {len(metadata):,}')
print(f'Số bệnh nhân duy nhất: {metadata["subject_id"].nunique():,}')
print(f'Số study duy nhất: {metadata["study_id"].nunique():,}')
metadata.head()


In [ ]:
# 2.3. Phân tích phân bố view position (nếu có)
# ViewPosition phổ biến: PA (Posteroanterior), AP (Anteroposterior), LATERAL, LL
# Trong nhiều nghiên cứu, người ta chỉ giữ view PA/AP (frontal) để training

if 'ViewPosition' in metadata.columns:
    view_counts = metadata['ViewPosition'].value_counts()
    print('Phân bố ViewPosition:')
    print(view_counts)

    fig, ax = plt.subplots(figsize=(10, 5))
    view_counts.head(10).plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Phân bố ViewPosition trong MIMIC-CXR')
    ax.set_xlabel('View Position')
    ax.set_ylabel('Số lượng ảnh')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('Không có cột ViewPosition - sẽ bỏ qua bước filter theo view')


In [ ]:
# 2.4. Quan sát một số ảnh mẫu

def show_sample_images(df, n=6):
    """Hiển thị n ảnh ngẫu nhiên kèm thông tin metadata."""
    samples = df.sample(n=min(n, len(df)), random_state=SEED)

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, (_, row) in zip(axes.flat, samples.iterrows()):
        try:
            img = Image.open(row['image_path'])
            ax.imshow(img, cmap='gray')
            ax.set_title(f'subj={row["subject_id"]}\nstudy={row["study_id"]}\nsize={img.size}', fontsize=9)
            ax.axis('off')
        except Exception as e:
            ax.text(0.5, 0.5, f'Error: {e}', ha='center')
            ax.axis('off')

    plt.tight_layout()
    plt.show()

show_sample_images(metadata, n=6)


## 3. Trích xuất section FINDINGS / IMPRESSION từ báo cáo

**Cấu trúc một báo cáo MIMIC-CXR điển hình:**
```
FINAL REPORT
 EXAMINATION: CHEST (PA AND LAT)
 INDICATION: ___ year old woman with cough
 TECHNIQUE: Chest PA and lateral
 COMPARISON: None
 FINDINGS: The cardiac silhouette is normal in size... 
 IMPRESSION: No acute cardiopulmonary process.
```

Hai section quan trọng nhất:
- **FINDINGS** — mô tả chi tiết quan sát của bác sĩ → thường dùng làm target cho report generation
- **IMPRESSION** — tóm tắt chẩn đoán → thường dùng cho summarization

In [ ]:
# 3.1. Hàm trích xuất section từ một báo cáo

def extract_sections(report_text):
    """Trích xuất các section chính từ báo cáo MIMIC-CXR.

    Trả về dict với keys: findings, impression, indication, comparison.
    Nếu không tìm thấy section nào, giá trị tương ứng sẽ là None.
    """
    sections = {
        'findings': None,
        'impression': None,
        'indication': None,
        'comparison': None
    }

    if not report_text or not isinstance(report_text, str):
        return sections

    # Danh sách các section header chuẩn (sẽ là biên để cắt FINDINGS)
    section_headers = [
        'FINAL REPORT', 'EXAMINATION:', 'INDICATION:', 'HISTORY:',
        'TECHNIQUE:', 'COMPARISON:', 'FINDINGS:', 'IMPRESSION:',
        'RECOMMENDATION:', 'NOTIFICATION:', 'CONCLUSION:'
    ]

    # Regex pattern cho từng section: bắt nội dung từ header tới header kế tiếp
    patterns = {
        'findings': r'FINDINGS:\s*(.+?)(?=' + '|'.join(section_headers) + r'|$)',
        'impression': r'IMPRESSION:\s*(.+?)(?=' + '|'.join(section_headers) + r'|$)',
        'indication': r'INDICATION:\s*(.+?)(?=' + '|'.join(section_headers) + r'|$)',
        'comparison': r'COMPARISON:\s*(.+?)(?=' + '|'.join(section_headers) + r'|$)'
    }

    for key, pattern in patterns.items():
        match = re.search(pattern, report_text, re.DOTALL | re.IGNORECASE)
        if match:
            content = match.group(1).strip()
            sections[key] = content if content else None

    return sections

# Test trên một báo cáo mẫu
sample_report = """ FINAL REPORT
 EXAMINATION: CHEST (PA AND LAT)
 INDICATION: ___ year old woman with ?pleural effusion
 TECHNIQUE: Chest PA and lateral
 COMPARISON: None
 FINDINGS: Cardiac size cannot be evaluated. Large left pleural
 effusion is new. Small right effusion is new. The upper lungs are clear.
 IMPRESSION: Large left pleural effusion.
"""

result = extract_sections(sample_report)
for k, v in result.items():
    print(f'{k.upper()}: {v}\n')


In [ ]:
# 3.2. Load báo cáo cho subset p10
# Báo cáo có thể ở dạng:
#   (a) Các file .txt riêng lẻ trong thư mục files/p<XX>/p<subject>/s<study>.txt
#   (b) Một file CSV tổng hợp đã được trích xuất sẵn

def load_reports_from_directory(base_path):
    """Đọc tất cả file .txt báo cáo và trả về DataFrame."""
    records = []
    txt_files = list(base_path.rglob('*.txt'))

    for txt_path in tqdm(txt_files, desc='Đọc báo cáo'):
        try:
            study_id = int(txt_path.stem.lstrip('s'))
            subject_id = int(txt_path.parent.name.lstrip('p'))

            with open(txt_path, 'r', encoding='utf-8') as f:
                content = f.read()

            records.append({
                'subject_id': subject_id,
                'study_id': study_id,
                'report_raw': content
            })
        except (ValueError, IOError):
            continue

    return pd.DataFrame(records)

def normalize_report_columns(df):
    out = df.copy()

    rename_map = {
        'Findings': 'findings',
        'FINDINGS': 'findings',
        'Impression': 'impression',
        'IMPRESSION': 'impression',
        'Report': 'report_raw',
        'REPORT': 'report_raw',
        'Reports': 'report_raw',
        'report_text': 'report_raw',
    }
    out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns and v not in out.columns})

    if 'subject_id' not in out.columns and 'Img_Folder' in out.columns:
        folders = out['Img_Folder'].astype(str)
        out['subject_id'] = folders.str.extract(r'(?:^|/)p(\d{8})(?:/|$)')[0].astype('Int64')

    if 'study_id' not in out.columns and 'Img_Folder' in out.columns:
        folders = out['Img_Folder'].astype(str)
        out['study_id'] = folders.str.extract(r'(?:^|/)s(\d+)(?:/|$)')[0].astype('Int64')

    if 'report_raw' not in out.columns:
        for col in ['report', 'Report', 'text', 'Text', 'raw_report', 'full_report']:
            if col in out.columns:
                out['report_raw'] = out[col]
                break

    out = out.dropna(subset=['subject_id', 'study_id']).copy()
    out['subject_id'] = out['subject_id'].astype(int)
    out['study_id'] = out['study_id'].astype(int)
    out = filter_to_patient_group(out, PATIENT_GROUP)

    # Một CSV image-level có thể có nhiều dòng cùng study. Giữ một report cho mỗi study.
    out = out.drop_duplicates(subset=['subject_id', 'study_id']).reset_index(drop=True)
    return out

# Thử load CSV tổng hợp trước, nếu không có thì scan thư mục p10
report_csv_path = find_csv_recursive(
    REPORT_DATASET_PATH,
    ['mimic_cxr_cleaned.csv', 'reports.csv', 'mimic_cxr_reports.csv', 'cxr_reports.csv'],
)

if report_csv_path:
    reports_df = pd.read_csv(report_csv_path)
    reports_df = normalize_report_columns(reports_df)
    print(f'Đã load báo cáo từ CSV: {report_csv_path}')
else:
    print('Không có CSV tổng hợp, scan thư mục .txt trong subset p10...')
    reports_df = load_reports_from_directory(PATIENT_GROUP_REPORT_PATH)
    reports_df = normalize_report_columns(reports_df)

print(f'Tổng số báo cáo trong {PATIENT_GROUP}: {len(reports_df):,}')
reports_df.head(2)


In [ ]:
# 3.3. Áp dụng hàm trích xuất section trên toàn bộ báo cáo
# Nếu CSV đã có sẵn findings/impression thì dùng trực tiếp.

if 'findings' not in reports_df.columns:
    if 'findings_clean' in reports_df.columns:
        reports_df['findings'] = reports_df['findings_clean']
    elif 'report_raw' in reports_df.columns:
        print('Đang trích xuất sections...')
        sections_df = reports_df['report_raw'].progress_apply(
            lambda x: pd.Series(extract_sections(x))
        )
        reports_df = pd.concat([reports_df, sections_df], axis=1)
    else:
        raise KeyError('Không có cột findings/findings_clean/report_raw để trích xuất FINDINGS.')

if 'impression' not in reports_df.columns:
    if 'impression_clean' in reports_df.columns:
        reports_df['impression'] = reports_df['impression_clean']
    elif 'report_raw' in reports_df.columns:
        print('Đang trích xuất sections cho IMPRESSION...')
        sections_df = reports_df['report_raw'].progress_apply(
            lambda x: pd.Series(extract_sections(x))
        )
        for col in sections_df.columns:
            if col not in reports_df.columns:
                reports_df[col] = sections_df[col]
    else:
        reports_df['impression'] = ''

# Thống kê tỷ lệ có/không có section
print('\nTỷ lệ báo cáo có chứa từng section:')
for col in ['findings', 'impression', 'indication', 'comparison']:
    if col in reports_df.columns:
        has_section = reports_df[col].notna() & (reports_df[col].astype(str).str.strip() != '')
        print(f'{col:12s}: {has_section.sum():>6,} / {len(reports_df):,} ({has_section.mean()*100:.1f}%)')


## 4. Làm sạch văn bản (Text Cleaning)

Các vấn đề thường gặp trong báo cáo MIMIC-CXR:
1. **De-identified tokens** `___` (3 dấu gạch dưới) thay cho thông tin cá nhân → chuẩn hóa thành `<UNK>`
2. **Xuống dòng giữa câu** do giới hạn 79 ký tự/dòng → cần ghép lại
3. **Khoảng trắng dư thừa, ký tự đặc biệt**
4. **Báo cáo quá ngắn (< 3 từ)** hoặc quá dài (> 300 từ) → cân nhắc loại bỏ

In [ ]:
def clean_text(text):
    """Làm sạch văn bản báo cáo y tế.

    Các bước:
    1. Thay token de-identified ___ bằng <UNK>
    2. Ghép các dòng bị ngắt (do giới hạn 79 ký tự/dòng)
    3. Chuẩn hóa khoảng trắng
    4. Lowercase
    5. Loại bỏ ký tự đặc biệt không cần thiết
    """
    if not text or not isinstance(text, str):
        return ''

    # 1. Thay token de-identified
    text = re.sub(r'_{2,}', '<UNK>', text)

    # 2. Loại bỏ newline (ghép các dòng bị ngắt)
    text = re.sub(r'\s*\n\s*', ' ', text)

    # 3. Chuẩn hóa nhiều khoảng trắng thành 1
    text = re.sub(r'\s+', ' ', text)

    # 4. Loại bỏ ký tự đặc biệt (giữ lại chữ, số, dấu câu cơ bản)
    text = re.sub(r'[^a-zA-Z0-9\s.,;:?!<>\-/()]', '', text)

    # 5. Lowercase và strip
    text = text.lower().strip()

    # 6. Chuẩn hóa khoảng trắng quanh dấu câu
    text = re.sub(r'\s+([.,;:?!])', r'\1', text)

    return text

# Test
dirty = """FINDINGS: The cardiac silhouette is normal in size.
 The lungs are clear without focal consolidation, pleural effusion or
 pneumothorax. Comparison was made with prior study of ___."""

print('Trước khi clean:')
print(repr(dirty))
print('\nSau khi clean:')
print(repr(clean_text(dirty)))


In [ ]:
# Áp dụng clean_text cho cột findings và impression
print('Đang làm sạch text...')

if 'findings_clean' not in reports_df.columns:
    reports_df['findings_clean'] = reports_df['findings'].progress_apply(clean_text)
else:
    reports_df['findings_clean'] = reports_df['findings_clean'].fillna('').astype(str)

if 'impression_clean' not in reports_df.columns:
    reports_df['impression_clean'] = reports_df['impression'].progress_apply(clean_text)
else:
    reports_df['impression_clean'] = reports_df['impression_clean'].fillna('').astype(str)

# Tính độ dài (số từ) sau khi clean
reports_df['findings_len'] = reports_df['findings_clean'].str.split().str.len().fillna(0).astype(int)
reports_df['impression_len'] = reports_df['impression_clean'].str.split().str.len().fillna(0).astype(int)

print('\nThống kê độ dài (số từ):')
print(reports_df[['findings_len', 'impression_len']].describe())


In [ ]:
# Trực quan hóa phân bố độ dài để chọn ngưỡng filter hợp lý
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(reports_df[reports_df['findings_len'] > 0]['findings_len'], bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Phân bố độ dài FINDINGS (số từ)')
axes[0].set_xlabel('Số từ')
axes[0].set_ylabel('Số báo cáo')
axes[0].axvline(reports_df['findings_len'].median(), color='red', linestyle='--', label=f'Median = {reports_df["findings_len"].median():.0f}')
axes[0].legend()

axes[1].hist(reports_df[reports_df['impression_len'] > 0]['impression_len'], bins=50, color='coral', edgecolor='black')
axes[1].set_title('Phân bố độ dài IMPRESSION (số từ)')
axes[1].set_xlabel('Số từ')
axes[1].set_ylabel('Số báo cáo')
axes[1].axvline(reports_df['impression_len'].median(), color='red', linestyle='--', label=f'Median = {reports_df["impression_len"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Ghép cặp Ảnh ↔ Báo cáo và Lọc dữ liệu

**Logic ghép cặp:** mỗi `study_id` tương ứng với 1 báo cáo và 1+ ảnh.
Sau khi merge, ta lọc bỏ các trường hợp:
- FINDINGS rỗng (không thể train model report generation)
- FINDINGS quá ngắn (< 5 từ) — báo cáo bất thường
- FINDINGS quá dài (> 200 từ) — outlier, có thể là báo cáo phức tạp khó học
- View không phải PA hoặc AP (chỉ giữ frontal view để giảm noise)

In [ ]:
# 5.1. Merge images với reports theo subject_id và study_id
reports_for_merge = reports_df[
    ['subject_id', 'study_id', 'findings_clean', 'impression_clean', 'findings_len', 'impression_len']
].drop_duplicates(subset=['subject_id', 'study_id'])

merged = metadata.merge(
    reports_for_merge,
    on=['subject_id', 'study_id'],
    how='inner'
)

print(f'Sau khi merge: {len(merged):,} cặp ảnh-báo cáo')
print(f'Trước khi merge: {len(metadata):,} ảnh, {len(reports_for_merge):,} báo cáo')
print(f'Số ảnh bị mất (không có báo cáo): {len(metadata) - len(merged):,}')
merged.head(2)


In [ ]:
# 5.2. Filter rules - có thể điều chỉnh theo bài toán cụ thể
MIN_FINDINGS_LEN = 5    # tối thiểu 5 từ
MAX_FINDINGS_LEN = 200  # tối đa 200 từ
KEEP_VIEWS = ['PA', 'AP']  # chỉ giữ frontal view

initial_count = len(merged)
filter_log = []

# Filter 1: Loại bỏ ảnh không có FINDINGS
merged = merged[merged['findings_clean'].notna() & (merged['findings_clean'] != '')]
filter_log.append(('Có FINDINGS', initial_count, len(merged)))

# Filter 2: Độ dài FINDINGS hợp lý
before = len(merged)
merged = merged[(merged['findings_len'] >= MIN_FINDINGS_LEN) &
                (merged['findings_len'] <= MAX_FINDINGS_LEN)]
filter_log.append((f'Độ dài FINDINGS trong [{MIN_FINDINGS_LEN}, {MAX_FINDINGS_LEN}]', before, len(merged)))

# Filter 3: View position (nếu có thông tin)
if 'ViewPosition' in merged.columns:
    before = len(merged)
    merged = merged[merged['ViewPosition'].isin(KEEP_VIEWS)]
    filter_log.append((f'View trong {KEEP_VIEWS}', before, len(merged)))

# In log chi tiết quá trình lọc
print('Quá trình lọc dữ liệu:')
print(f'{"Bước":<45} {"Trước":>10} {"Sau":>10} {"Mất":>8}')
print('-' * 75)
for step, before, after in filter_log:
    loss = before - after
    print(f'{step:<45} {before:>10,} {after:>10,} {loss:>8,}')
print('-' * 75)
print(f'{"TỔNG KẾT":<45} {initial_count:>10,} {len(merged):>10,} {initial_count-len(merged):>8,}')
print(f'\nTỷ lệ giữ lại: {len(merged)/initial_count*100:.1f}%')


In [ ]:
# 5.3. Kiểm tra integrity của file ảnh (loại bỏ ảnh lỗi/không đọc được)
# Lưu ý: bước này tốn thời gian, có thể skip nếu dataset đã được verify trước đó

CHECK_IMAGE_INTEGRITY = False  # đổi thành True nếu muốn verify

def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

if CHECK_IMAGE_INTEGRITY:
    print('Kiểm tra tính toàn vẹn của ảnh...')
    merged['is_valid'] = merged['image_path'].progress_apply(is_valid_image)
    invalid_count = (~merged['is_valid']).sum()
    print(f'Số ảnh lỗi: {invalid_count}')
    merged = merged[merged['is_valid']].drop(columns='is_valid')
    print(f'Còn lại sau khi loại ảnh lỗi: {len(merged):,}')
else:
    print('Bỏ qua kiểm tra integrity (CHECK_IMAGE_INTEGRITY=False)')


## 6. Xử lý ảnh (Image Preprocessing)

**Các bước chuẩn cho ảnh X-quang khi đưa vào CNN:**
1. **Resize** về kích thước cố định (thường 224×224 cho ResNet/DenseNet, 384×384 cho Swin Transformer)
2. **Convert sang grayscale** (X-quang vốn là ảnh xám, nhưng nhiều model pretrained cần 3 channels → duplicate channel)
3. **Normalize** theo mean/std của ImageNet (nếu dùng pretrained model)
4. **Caching**: lưu ảnh đã resize ra disk dạng numpy/tensor để tăng tốc training (tuỳ chọn)

Trong notebook này, ta chỉ **chuẩn hóa metadata về ảnh** (path, kích thước) và để bước transform thực tế xảy ra trong DataLoader khi training.

In [ ]:
# 6.1. Định nghĩa cấu hình ảnh
IMG_SIZE = 224  # kích thước resize

# 6.2. Lấy thông tin kích thước gốc của một mẫu để báo cáo
sample_paths = merged['image_path'].sample(n=min(100, len(merged)), random_state=SEED).tolist()
sizes = []
for p in sample_paths:
    try:
        with Image.open(p) as img:
            sizes.append(img.size)
    except Exception:
        continue

if sizes:
    widths, heights = zip(*sizes)
    print(f'Thống kê kích thước ảnh gốc (n={len(sizes)} mẫu):')
    print(f'Width: min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}')
    print(f'Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}')
    print(f'\nResize target: {IMG_SIZE}x{IMG_SIZE}')


In [ ]:
# 6.3. Code mẫu cho transform pipeline khi training (để tham khảo)
# Đây là transform PyTorch sẽ apply trong DataLoader, KHÔNG chạy ở preprocessing

transform_code = '''
# === Tham khảo: dùng trong training script ===
from torchvision import transforms

# Transform cho training (có augmentation)
train_transform = transforms.Compose([
 transforms.Resize((256, 256)),
 transforms.RandomCrop(224),
 transforms.RandomHorizontalFlip(p=0.5),
 transforms.RandomRotation(degrees=10),
 transforms.Grayscale(num_output_channels=3), # convert 1-ch → 3-ch để dùng pretrained
 transforms.ToTensor(),
 transforms.Normalize(mean=[0.485, 0.456, 0.406], # ImageNet stats
 std=[0.229, 0.224, 0.225])
])

# Transform cho val/test (KHÔNG augmentation)
eval_transform = transforms.Compose([
 transforms.Resize((224, 224)),
 transforms.Grayscale(num_output_channels=3),
 transforms.ToTensor(),
 transforms.Normalize(mean=[0.485, 0.456, 0.406],
 std=[0.229, 0.224, 0.225])
])
'''
print(transform_code)

## 7. Tokenization & Vocabulary Building

Để model học sinh báo cáo, ta cần biến text thành sequence of token IDs.

**2 hướng tiếp cận:**
- **Word-level tokenization** + vocabulary tự build (đơn giản, dùng cho LSTM/Transformer from-scratch)
- **Subword tokenization** (BPE/WordPiece) từ pretrained tokenizer như BioClinical-BERT, RadBERT (tốt hơn cho domain y khoa)

Notebook này demo cả 2 cách.

In [ ]:
# 7.1. Cách 1: Word-level tokenization với vocab tự build

# Token đặc biệt
SPECIAL_TOKENS = {
    '<PAD>': 0,   # padding
    '<BOS>': 1,   # bắt đầu câu (Beginning of Sentence)
    '<EOS>': 2,   # kết thúc câu
    '<UNK>': 3,   # từ không có trong vocab
}

MIN_WORD_FREQ = 5  # từ phải xuất hiện ít nhất 5 lần mới được vào vocab

def build_vocabulary(text_series, min_freq=5):
    """Xây vocabulary từ corpus text."""
    counter = Counter()
    for text in tqdm(text_series, desc='Đếm từ'):
        if isinstance(text, str):
            counter.update(text.split())

    # Bắt đầu với special tokens
    word2idx = dict(SPECIAL_TOKENS)
    idx = len(SPECIAL_TOKENS)

    # Thêm các từ có frequency >= min_freq
    for word, freq in counter.most_common():
        if freq >= min_freq and word not in word2idx:
            word2idx[word] = idx
            idx += 1

    idx2word = {v: k for k, v in word2idx.items()}

    return word2idx, idx2word, counter

word2idx, idx2word, word_counter = build_vocabulary(merged['findings_clean'], min_freq=MIN_WORD_FREQ)

print(f'\nVocabulary stats:')
print(f'Tổng số từ duy nhất trong corpus: {len(word_counter):,}')
print(f'Vocabulary size (min_freq={MIN_WORD_FREQ}): {len(word2idx):,}')
print('Top 20 từ phổ biến nhất:')
for word, freq in word_counter.most_common(20):
    print(f'  {word:20s} {freq:>8,}')


In [ ]:
# 7.2. Hàm encode/decode text -> sequence of IDs

MAX_SEQ_LEN = 100  # độ dài tối đa của sequence sau tokenize (gồm <BOS> và <EOS>)

def encode_text(text, word2idx, max_len=MAX_SEQ_LEN):
    """Convert text -> list of token IDs với padding."""
    if not isinstance(text, str):
        return [word2idx['<PAD>']] * max_len

    tokens = ['<BOS>'] + text.split() + ['<EOS>']
    ids = [word2idx.get(tok, word2idx['<UNK>']) for tok in tokens]

    # Truncate hoặc pad
    if len(ids) > max_len:
        ids = ids[:max_len-1] + [word2idx['<EOS>']]  # vẫn giữ EOS ở cuối
    else:
        ids = ids + [word2idx['<PAD>']] * (max_len - len(ids))

    return ids

def decode_ids(ids, idx2word, skip_special=True):
    """Convert list of IDs -> text."""
    special_ids = set(SPECIAL_TOKENS.values()) if skip_special else set()
    tokens = [idx2word.get(i, '<UNK>') for i in ids if i not in special_ids]
    return ' '.join(tokens)

# Test
sample = merged['findings_clean'].iloc[0]
print(f'Original: {sample[:150]}...\n')
encoded = encode_text(sample, word2idx)
print(f'Encoded (first 30 IDs): {encoded[:30]}\n')
print(f'Decoded: {decode_ids(encoded, idx2word)[:150]}...')


In [ ]:
# 7.3. Cách 2 (tuỳ chọn): Subword tokenization với HuggingFace tokenizer
# Bỏ comment để dùng — yêu cầu kết nối internet hoặc cài transformers offline

USE_HF_TOKENIZER = False # Đổi True nếu muốn dùng pretrained tokenizer

if USE_HF_TOKENIZER:
 from transformers import AutoTokenizer
 
 # Các tokenizer phù hợp domain y khoa:
 # - 'emilyalsentzer/Bio_ClinicalBERT' — clinical text
 # - 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext'
 # - 'StanfordAIMI/RadBERT' — radiology reports
 
 hf_tokenizer = AutoTokenizer.from_pretrained('emilyalsentzer/Bio_ClinicalBERT')
 
 sample_text = merged['findings_clean'].iloc[0]
 tokens = hf_tokenizer(sample_text, padding='max_length', max_length=MAX_SEQ_LEN, 
 truncation=True, return_tensors='np')
 print(f' Input IDs shape: {tokens["input_ids"].shape}')
 print(f' First 30 tokens: {hf_tokenizer.convert_ids_to_tokens(tokens["input_ids"][0][:30])}')
else:
 print(' Bỏ qua HuggingFace tokenizer (USE_HF_TOKENIZER=False)')

## 8. Chia tập Train/Val/Test, đánh giá phân bố nhãn và lưu kết quả

**Nguyên tắc quan trọng:** Split theo `subject_id` để tránh data leakage. Một bệnh nhân không được xuất hiện đồng thời ở train/val/test.

Notebook này chỉ split dữ liệu thuộc `PATIENT_GROUP = "p10"` và lưu thêm file `mimic-cxr-2.0.0-split-p10.csv`.


In [ ]:
# 8.1. Split subset p10 theo subject_id
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

unique_subjects = merged['subject_id'].unique()
print(f'Tổng số bệnh nhân duy nhất trong {PATIENT_GROUP}: {len(unique_subjects):,}')

# Bước 1: tách test set ra trước
train_val_subjects, test_subjects = train_test_split(
    unique_subjects, test_size=TEST_RATIO, random_state=SEED
)

# Bước 2: tách val từ phần train+val còn lại
val_size_adjusted = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_subjects, val_subjects = train_test_split(
    train_val_subjects, test_size=val_size_adjusted, random_state=SEED
)

# Áp dụng split vào DataFrame
merged['split'] = 'train'
merged.loc[merged['subject_id'].isin(val_subjects), 'split'] = 'val'
merged.loc[merged['subject_id'].isin(test_subjects), 'split'] = 'test'

# Thống kê
print('\nPhân bố sau khi split:')
split_stats = merged['split'].value_counts()
for split_name in ['train', 'val', 'test']:
    count = split_stats.get(split_name, 0)
    subjects = merged[merged['split'] == split_name]['subject_id'].nunique()
    pct = count / len(merged) * 100
    print(f'  {split_name:5s}: {count:>8,} ảnh ({pct:.1f}%) | {subjects:>6,} bệnh nhân')

# Kiểm tra không có patient leakage
train_set = set(merged[merged['split'] == 'train']['subject_id'])
val_set = set(merged[merged['split'] == 'val']['subject_id'])
test_set = set(merged[merged['split'] == 'test']['subject_id'])

assert len(train_set & val_set) == 0, 'LEAKAGE: train và val có chung subject!'
assert len(train_set & test_set) == 0, 'LEAKAGE: train và test có chung subject!'
assert len(val_set & test_set) == 0, 'LEAKAGE: val và test có chung subject!'
print('\nKhông có data leakage giữa các split')


In [ ]:
# 8.2. Đánh giá lại phân bố nhãn P/N/U theo split cho subset p10
# Có hai cách thống kê:
#   1. Explicit labels: chỉ tính các giá trị CheXpert có sẵn P=1, N=0, U=-1.
#   2. Training labels: giống dataloader ReportDataset, tức U=-1 -> 2 và NaN -> 0 (negative).

LABEL_COLUMNS = [
    'Atelectasis',
    'Cardiomegaly',
    'Consolidation',
    'Edema',
    'Enlarged Cardiomediastinum',
    'Fracture',
    'Lung Lesion',
    'Lung Opacity',
    'No Finding',
    'Pleural Effusion',
    'Pleural Other',
    'Pneumonia',
    'Pneumothorax',
    'Support Devices',
]

LABEL_DISPLAY_NAMES = {
    'Enlarged Cardiomediastinum': 'Enlarged Cardio',
}

chexpert_path = find_csv_recursive(
    IMAGE_DATASET_PATH,
    ['mimic-cxr-2.0.0-chexpert.csv', 'chexpert.csv'],
)

if chexpert_path is None:
    print('Không tìm thấy file CheXpert label CSV, bỏ qua bảng phân bố P/N/U.')
    label_count_table = pd.DataFrame()
    label_ratio_table = pd.DataFrame()
    label_missing_table = pd.DataFrame()
    training_label_count_table = pd.DataFrame()
    training_label_ratio_table = pd.DataFrame()
    training_label_overall_table = pd.DataFrame()
else:
    chexpert = pd.read_csv(chexpert_path)
    chexpert = filter_to_patient_group(chexpert, PATIENT_GROUP)

    available_labels = [col for col in LABEL_COLUMNS if col in chexpert.columns]
    if not available_labels:
        raise ValueError(f'Không tìm thấy cột nhãn CheXpert trong {chexpert_path}')

    label_source = chexpert[['subject_id', 'study_id'] + available_labels].drop_duplicates(
        subset=['subject_id', 'study_id']
    )
    split_source = merged[['dicom_id', 'subject_id', 'study_id', 'split']].drop_duplicates(
        subset=['dicom_id']
    )
    split_labels = split_source.merge(label_source, on=['subject_id', 'study_id'], how='left')

    split_order = ['train', 'test', 'val']
    explicit_count_rows = []
    explicit_ratio_rows = []
    missing_rows = []
    training_count_rows = []
    training_ratio_rows = []
    overall_training_rows = []

    for label in available_labels:
        display_name = LABEL_DISPLAY_NAMES.get(label, label)
        explicit_count_row = {'Class': display_name}
        explicit_ratio_row = {'Class': display_name}
        missing_row = {'Class': display_name}
        training_count_row = {'Class': display_name}
        training_ratio_row = {'Class': display_name}

        for split_name in split_order:
            values = split_labels.loc[split_labels['split'] == split_name, label]

            # Bảng explicit: mẫu số chỉ gồm các nhãn CheXpert đã có giá trị rõ ràng.
            total_explicit = int(values.isin([1, 0, -1]).sum())
            for status_name, status_value in [('P', 1), ('N', 0), ('U', -1)]:
                count = int((values == status_value).sum())
                explicit_count_row[(split_name.title(), status_name)] = count
                explicit_ratio_row[(split_name.title(), status_name)] = (
                    round(count / total_explicit * 100, 2) if total_explicit else 0.0
                )

            missing_count = int(values.isna().sum())
            split_total = int(len(values))
            missing_row[(split_name.title(), 'Missing')] = missing_count
            missing_row[(split_name.title(), 'Missing %')] = (
                round(missing_count / split_total * 100, 2) if split_total else 0.0
            )

            # Bảng training: giống ReportDataset, -1 thành uncertain class 2, NaN thành negative class 0.
            train_values = values.replace(-1, 2).fillna(0).astype(int)
            for status_name, status_value in [('P', 1), ('N', 0), ('U', 2)]:
                count = int((train_values == status_value).sum())
                training_count_row[(split_name.title(), status_name)] = count
                training_ratio_row[(split_name.title(), status_name)] = (
                    round(count / split_total * 100, 2) if split_total else 0.0
                )

        explicit_count_rows.append(explicit_count_row)
        explicit_ratio_rows.append(explicit_ratio_row)
        missing_rows.append(missing_row)
        training_count_rows.append(training_count_row)
        training_ratio_rows.append(training_ratio_row)

    label_count_table = pd.DataFrame(explicit_count_rows)
    label_ratio_table = pd.DataFrame(explicit_ratio_rows)
    label_missing_table = pd.DataFrame(missing_rows)
    training_label_count_table = pd.DataFrame(training_count_rows)
    training_label_ratio_table = pd.DataFrame(training_ratio_rows)

    for split_name in split_order:
        split_subset = split_labels.loc[split_labels['split'] == split_name, available_labels]
        train_subset = split_subset.replace(-1, 2).fillna(0).astype(int)
        total_labels = int(train_subset.size)
        p_count = int((train_subset == 1).sum().sum())
        n_count = int((train_subset == 0).sum().sum())
        u_count = int((train_subset == 2).sum().sum())
        overall_training_rows.append({
            'Split': split_name,
            'Images': int(len(train_subset)),
            'Total labels': total_labels,
            'P': p_count,
            'N': n_count,
            'U': u_count,
            'P %': round(p_count / total_labels * 100, 2) if total_labels else 0.0,
            'N %': round(n_count / total_labels * 100, 2) if total_labels else 0.0,
            'U %': round(u_count / total_labels * 100, 2) if total_labels else 0.0,
        })
    training_label_overall_table = pd.DataFrame(overall_training_rows)

    def class_first(df):
        value_cols = [col for col in df.columns if col != 'Class']
        return df[['Class'] + value_cols]

    label_count_table = class_first(label_count_table)
    label_ratio_table = class_first(label_ratio_table)
    label_missing_table = class_first(label_missing_table)
    training_label_count_table = class_first(training_label_count_table)
    training_label_ratio_table = class_first(training_label_ratio_table)

    # File cũ giữ nguyên ý nghĩa explicit-only để không phá workflow đang dùng.
    label_count_table.to_csv(OUTPUT_DIR / f'{PATIENT_GROUP}_label_distribution_counts.csv', index=False)
    label_ratio_table.to_csv(OUTPUT_DIR / f'{PATIENT_GROUP}_label_distribution_ratios.csv', index=False)

    # File mới phản ánh đúng label mà training sẽ nhận.
    label_missing_table.to_csv(OUTPUT_DIR / f'{PATIENT_GROUP}_label_missing_counts.csv', index=False)
    training_label_count_table.to_csv(OUTPUT_DIR / f'{PATIENT_GROUP}_training_label_distribution_counts.csv', index=False)
    training_label_ratio_table.to_csv(OUTPUT_DIR / f'{PATIENT_GROUP}_training_label_distribution_ratios.csv', index=False)
    training_label_overall_table.to_csv(OUTPUT_DIR / f'{PATIENT_GROUP}_training_label_distribution_overall.csv', index=False)

    print(f'CheXpert labels: {chexpert_path}')
    print(f'Tổng số ảnh có split dùng để thống kê: {len(split_labels):,}')

    print('\n[1] Số lượng nhãn explicit P/N/U từ CheXpert, bỏ qua NaN:')
    display(label_count_table)
    print('\n[1] Tỷ lệ nhãn explicit P/N/U từ CheXpert, bỏ qua NaN (%):')
    display(label_ratio_table)

    print('\nSố lượng nhãn thiếu NaN theo từng split (các NaN này sẽ thành negative khi train):')
    display(label_missing_table)

    print('\n[2] Số lượng nhãn thực tế khi train: U=-1 -> 2, NaN -> N=0')
    display(training_label_count_table)
    print('\n[2] Tỷ lệ nhãn thực tế khi train (%):')
    display(training_label_ratio_table)

    print('\nTổng hợp toàn bộ 14 nhãn sau mapping dùng cho training:')
    display(training_label_overall_table)


In [ ]:
# 8.3. Tokenize toàn bộ và lưu kèm vào DataFrame
print('Đang tokenize toàn bộ FINDINGS...')
merged['findings_ids'] = merged['findings_clean'].progress_apply(
    lambda x: encode_text(x, word2idx, MAX_SEQ_LEN)
)

# Verify một mẫu
sample_row = merged.iloc[0]
print(f'\nSample findings: {sample_row["findings_clean"][:100]}...')
print(f'Encoded length: {len(sample_row["findings_ids"])}')
print(f'First 20 IDs: {sample_row["findings_ids"][:20]}')


In [ ]:
# 8.4. Lưu các artifact ra disk

# Lưu DataFrame chính (chỉ giữ cột cần thiết)
output_cols = [
    'dicom_id',
    'subject_id',
    'study_id',
    'image_path',
    'findings_clean',
    'impression_clean',
    'findings_len',
    'split',
]
output_cols = [c for c in output_cols if c in merged.columns]

final_df = merged[output_cols].copy()
final_df = final_df.sort_values(['split', 'subject_id', 'study_id', 'dicom_id']).reset_index(drop=True)

# Lưu CSV cho từng split. Ghi cả tên chuẩn và tên có prefix p10 để dễ upload riêng.
for split_name in ['train', 'val', 'test']:
    split_df = final_df[final_df['split'] == split_name]
    out_path = OUTPUT_DIR / f'{split_name}.csv'
    p10_out_path = OUTPUT_DIR / f'{PATIENT_GROUP}_{split_name}.csv'
    split_df.to_csv(out_path, index=False)
    split_df.to_csv(p10_out_path, index=False)
    print(f'Saved {out_path.name}: {len(split_df):,} rows')
    print(f'Saved {p10_out_path.name}: {len(split_df):,} rows')

# Lưu file tổng hợp
final_df.to_csv(OUTPUT_DIR / 'all_data.csv', index=False)
final_df.to_csv(OUTPUT_DIR / f'{PATIENT_GROUP}_all_data.csv', index=False)
print(f'Saved all_data.csv: {len(final_df):,} rows')
print(f'Saved {PATIENT_GROUP}_all_data.csv: {len(final_df):,} rows')

# Lưu split CSV mới chỉ cho subset p10, cùng schema chính của MIMIC split CSV.
split_output_cols = [c for c in ['dicom_id', 'study_id', 'subject_id', 'split'] if c in final_df.columns]
p10_split_df = final_df[split_output_cols].drop_duplicates(subset=['dicom_id']).copy()
p10_split_path = OUTPUT_DIR / f'mimic-cxr-2.0.0-split-{PATIENT_GROUP}.csv'
p10_split_df.to_csv(p10_split_path, index=False)
print(f'Saved {p10_split_path.name}: {len(p10_split_df):,} rows')

# Lưu vocabulary (pickle)
vocab_artifacts = {
    'word2idx': word2idx,
    'idx2word': idx2word,
    'special_tokens': SPECIAL_TOKENS,
    'max_seq_len': MAX_SEQ_LEN,
    'min_word_freq': MIN_WORD_FREQ,
    'vocab_size': len(word2idx)
}
with open(OUTPUT_DIR / 'vocabulary.pkl', 'wb') as f:
    pickle.dump(vocab_artifacts, f)
print(f'Saved vocabulary.pkl ({len(word2idx):,} tokens)')

# Lưu config dưới dạng JSON để dễ đọc
config = {
    'preprocessing_config': {
        'patient_group': PATIENT_GROUP,
        'subject_prefix': SUBJECT_PREFIX,
        'min_findings_length': MIN_FINDINGS_LEN,
        'max_findings_length': MAX_FINDINGS_LEN,
        'keep_views': KEEP_VIEWS,
        'image_size': IMG_SIZE,
        'max_seq_len': MAX_SEQ_LEN,
        'min_word_freq': MIN_WORD_FREQ,
        'vocab_size': len(word2idx),
        'random_seed': SEED
    },
    'split_ratios': {
        'train': TRAIN_RATIO,
        'val': VAL_RATIO,
        'test': TEST_RATIO
    },
    'dataset_stats': {
        'total_samples': len(final_df),
        'total_subjects': final_df['subject_id'].nunique(),
        'train_samples': int((final_df['split'] == 'train').sum()),
        'val_samples': int((final_df['split'] == 'val').sum()),
        'test_samples': int((final_df['split'] == 'test').sum())
    },
    'outputs': {
        'split_csv': str(p10_split_path),
        'output_dir': str(OUTPUT_DIR)
    }
}
with open(OUTPUT_DIR / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print('Saved config.json')

print(f'\nHoàn thành. Tất cả file output ở: {OUTPUT_DIR}')
print('\nDanh sách file output:')
for f in sorted(OUTPUT_DIR.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'   - {f.name:45s} {size_mb:>8.2f} MB')


## 9. Sanity Check — Kiểm tra cuối cùng

Trước khi đưa vào training, kiểm tra lại một số mẫu để chắc chắn pipeline chạy đúng.

In [ ]:
# Load lại từ file đã lưu để verify
train_df = pd.read_csv(OUTPUT_DIR / 'train.csv')
with open(OUTPUT_DIR / 'vocabulary.pkl', 'rb') as f:
    vocab = pickle.load(f)

print(f'Loaded train set: {len(train_df):,} samples')
print(f'Loaded vocabulary: {vocab["vocab_size"]:,} tokens')

# Hiển thị 3 mẫu ngẫu nhiên kèm ảnh
sample = train_df.sample(n=min(3, len(train_df)), random_state=SEED)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = np.atleast_1d(axes)

for ax, (_, row) in zip(axes, sample.iterrows()):
    try:
        img = Image.open(row['image_path'])
        ax.imshow(img, cmap='gray')
        title = f'subj={row["subject_id"]}, study={row["study_id"]}'
        findings_short = row['findings_clean'][:150] + ('...' if len(row['findings_clean']) > 150 else '')
        ax.set_title(title, fontsize=10)
        ax.set_xlabel(findings_short, fontsize=8, wrap=True)
        ax.set_xticks([])
        ax.set_yticks([])
    except Exception as e:
        ax.text(0.5, 0.5, f'Error: {e}', ha='center')

for ax in axes[len(sample):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

print('\nPipeline tiền xử lý hoàn tất, sẵn sàng cho bước training.')


## Tổng kết và bước tiếp theo

**Đã hoàn thành:**
- Load và parse cấu trúc dataset MIMIC-CXR
- Lọc dữ liệu về patient group `p10`
- Trích xuất FINDINGS / IMPRESSION từ báo cáo thô hoặc dùng CSV đã tiền xử lý
- Làm sạch text
- Ghép cặp ảnh và báo cáo theo `subject_id`, `study_id`
- Lọc dữ liệu theo độ dài findings và view position
- Xây vocabulary và tokenize
- Chia train/val/test theo `subject_id` để tránh leakage
- Đánh giá lại phân bố nhãn P/N/U theo split, gồm cả bảng explicit-only và bảng label thực tế dùng khi train (`NaN -> negative`)
- Lưu artifact ra disk

**File output sẵn sàng cho training:**
- `train.csv`, `val.csv`, `test.csv` - metadata + đường dẫn ảnh + findings
- `p10_train.csv`, `p10_val.csv`, `p10_test.csv` - bản có prefix subset
- `mimic-cxr-2.0.0-split-p10.csv` - split CSV mới chỉ cho subset `p10`
- `p10_label_distribution_counts.csv` - số lượng P/N/U theo split
- `p10_label_distribution_ratios.csv` - tỷ lệ P/N/U explicit theo split, bỏ qua NaN
- `p10_label_missing_counts.csv` - số lượng label NaN theo split
- `p10_training_label_distribution_counts.csv` - số lượng label thực tế lúc train (`-1 -> U`, `NaN -> N`)
- `p10_training_label_distribution_ratios.csv` - tỷ lệ label thực tế lúc train
- `p10_training_label_distribution_overall.csv` - tổng P/N/U toàn bộ 14 nhãn theo split
- `vocabulary.pkl` - word2idx / idx2word mapping
- `config.json` - toàn bộ cấu hình preprocessing

**Bước tiếp theo (training):**
1. Trỏ training config tới `mimic-cxr-2.0.0-split-p10.csv` nếu model đang đọc split CSV riêng.
2. Dùng các CSV trong `/kaggle/working/processed/p10` cho bước train/val/test.
3. Evaluate bằng metrics phù hợp: BLEU, ROUGE, METEOR, CIDEr, BERTScore hoặc classification metrics nếu train multi-label.


In [ ]:
# 10. So sánh số ảnh train giữa split gốc và bản train đã xử lý của bạn
# Mặc định so sánh theo ViewPosition (PA/AP/LATERAL/LL/...), tức loại view của ảnh X-quang.

from pathlib import Path

# ---- Load nguồn dữ liệu cần so sánh ----
original_split_path = find_csv_recursive(
    IMAGE_DATASET_PATH,
    ['mimic-cxr-2.0.0-split.csv', 'split.csv', 'mimic_cxr_split.csv'],
)

if original_split_path is None:
    raise FileNotFoundError('Không tìm thấy split CSV gốc trong IMAGE_DATASET_PATH.')

original_split = pd.read_csv(original_split_path)
original_split['split'] = original_split['split'].astype(str).str.lower()

# metadata_all đã được tạo ở cell 2.1 nếu có metadata_path; fallback để cell vẫn chạy được khi cần.
if 'metadata_all' not in globals():
    if 'metadata_path' not in globals() or metadata_path is None:
        raise FileNotFoundError('Không tìm thấy metadata gốc để xác định ViewPosition.')
    metadata_all = pd.read_csv(metadata_path)

metadata_keys = [c for c in ['dicom_id', 'study_id', 'subject_id'] if c in metadata_all.columns]
metadata_view_cols = metadata_keys + [c for c in ['ViewPosition'] if c in metadata_all.columns]
if 'ViewPosition' not in metadata_view_cols:
    raise KeyError('Metadata không có cột ViewPosition để đếm theo loại ảnh.')

metadata_view = metadata_all[metadata_view_cols].drop_duplicates(subset=metadata_keys)

# Split gốc: lấy train toàn bộ MIMIC-CXR và thêm một lát cắt chỉ riêng PATIENT_GROUP để so sánh công bằng với p10.
original_train = original_split[original_split['split'] == 'train'].copy()
merge_keys = [c for c in ['dicom_id', 'study_id'] if c in original_train.columns and c in metadata_view.columns]
if not merge_keys:
    raise KeyError('Không có khóa chung giữa split gốc và metadata để merge.')

original_train = original_train.merge(metadata_view, on=merge_keys, how='left', suffixes=('', '_meta'))
if 'subject_id' not in original_train.columns and 'subject_id_meta' in original_train.columns:
    original_train['subject_id'] = original_train['subject_id_meta']

original_train_pgroup = filter_to_patient_group(original_train, PATIENT_GROUP)

# Bản train của bạn: ưu tiên final_df đang có trong notebook, nếu không thì đọc CSV đã lưu.
if 'final_df' in globals():
    my_train = final_df[final_df['split'].astype(str).str.lower() == 'train'].copy()
else:
    train_candidates = [OUTPUT_DIR / f'{PATIENT_GROUP}_train.csv', OUTPUT_DIR / 'train.csv']
    train_path = next((p for p in train_candidates if Path(p).exists()), None)
    if train_path is None:
        raise FileNotFoundError(f'Không tìm thấy train CSV trong {OUTPUT_DIR}. Hãy chạy cell lưu artifact trước.')
    my_train = pd.read_csv(train_path)
    if 'split' in my_train.columns:
        my_train = my_train[my_train['split'].astype(str).str.lower() == 'train'].copy()

# Ghép ViewPosition vào bản train của bạn nếu file đã xử lý chưa có cột này.
if 'ViewPosition' not in my_train.columns:
    my_merge_keys = [c for c in ['dicom_id', 'study_id', 'subject_id'] if c in my_train.columns and c in metadata_view.columns]
    if not my_merge_keys:
        my_merge_keys = [c for c in ['dicom_id', 'study_id'] if c in my_train.columns and c in metadata_view.columns]
    if not my_merge_keys:
        raise KeyError('Không có khóa chung giữa train CSV của bạn và metadata để merge ViewPosition.')
    my_train = my_train.merge(metadata_view, on=my_merge_keys, how='left', suffixes=('', '_meta'))
    if 'ViewPosition' not in my_train.columns and 'ViewPosition_meta' in my_train.columns:
        my_train['ViewPosition'] = my_train['ViewPosition_meta']

# ---- Hàm thống kê ----
def normalize_view(series):
    return series.fillna('UNKNOWN').astype(str).str.strip().str.upper().replace({'': 'UNKNOWN', 'NAN': 'UNKNOWN'})

def summarize_train_images(df, source_name):
    out = df.copy()
    out['image_type'] = normalize_view(out['ViewPosition'])
    group = out.groupby('image_type', dropna=False)

    summary = group.agg(
        image_count=('dicom_id', 'nunique') if 'dicom_id' in out.columns else ('image_type', 'size'),
        row_count=('image_type', 'size'),
        study_count=('study_id', 'nunique') if 'study_id' in out.columns else ('image_type', 'size'),
        subject_count=('subject_id', 'nunique') if 'subject_id' in out.columns else ('image_type', 'size'),
    ).reset_index()
    summary.insert(0, 'source', source_name)
    summary = summary.sort_values(['image_count', 'image_type'], ascending=[False, True]).reset_index(drop=True)
    return summary

summary_tables = [
    summarize_train_images(original_train, 'Gốc - train toàn bộ'),
    summarize_train_images(original_train_pgroup, f'Gốc - train {PATIENT_GROUP}'),
    summarize_train_images(my_train, f'Của tôi - train {PATIENT_GROUP}'),
]
train_view_summary = pd.concat(summary_tables, ignore_index=True)

print(f'Split gốc: {original_split_path}')
print(f'Patient group đang dùng: {PATIENT_GROUP}')
print('\nTổng số ảnh train:')
for source_name, df in [
    ('Gốc - train toàn bộ', original_train),
    (f'Gốc - train {PATIENT_GROUP}', original_train_pgroup),
    (f'Của tôi - train {PATIENT_GROUP}', my_train),
]:
    image_total = df['dicom_id'].nunique() if 'dicom_id' in df.columns else len(df)
    study_total = df['study_id'].nunique() if 'study_id' in df.columns else 0
    subject_total = df['subject_id'].nunique() if 'subject_id' in df.columns else 0
    print(f'  {source_name:24s}: {image_total:>9,} ảnh | {study_total:>8,} study | {subject_total:>7,} bệnh nhân')

print('\nSố ảnh train theo loại view:')
display(train_view_summary)

train_view_pivot = train_view_summary.pivot_table(
    index='image_type', columns='source', values='image_count', fill_value=0, aggfunc='sum'
).astype(int)
train_view_pivot = train_view_pivot.sort_values(by=f'Của tôi - train {PATIENT_GROUP}', ascending=False)

print('\nBảng pivot số ảnh theo loại view:')
display(train_view_pivot)
